In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import CRUD module
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "fghijkl1$"
hostname = 'localhost' 
port = 27017 
database = 'aac' 
collection = 'animals'

# Connect to database via CRUD Module
db = AnimalShelter(username, password, hostname, port, database, collection)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' 
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Hr(),
    
    # Grazioso Salvare logo
    html.Div([
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()), 
            style={'width': '120px', 'display': 'block', 'margin': 'auto'}
        )
    ], style={'textAlign': 'center'}),  # Center the image
    
    # Unique identifier 
    html.Div(style={'borderTop': '10px solid #FFB6C1', 'margin': '20px 0'}),
    html.H1("Welcome to The Rescue Dashboard 🐾, I'm Zuha!", 
            style={'textAlign': 'center', 'color': '#FFB6C1'}),
    html.Hr(),
    
    # Radio buttons to select the rescue type
    html.Div(
        children=[
            html.Label("Select Rescue Type:", style={'fontSize': '18px', 'paddingRight': '10px'}),  # Label for radio buttons
            dcc.RadioItems(
                id='filter-type',  # Unique ID for radio button
                options=[  # Options for radio buttons
                    {'label': 'Water Rescue', 'value': 'water'},
                    {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                    {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
                    {'label': 'Reset', 'value': 'reset'},  # Default option to reset all filters
                ],
                value='reset',  # Default value when the page loads (Reset)
                labelStyle={'display': 'inline-block', 'fontSize': '16px', 'marginRight': '20px'},
                style={  
                    'padding': '10px',  
                    'user-select': 'none',  
                    'pointer-events': 'auto', 
                },
            ),
        ],
        style={  # Flexbox layout for alignment
            'display': 'flex',  
            'alignItems': 'center',  
            'justifyContent': 'left',  
            'gap': '10px',  
            'margin': '10px 0', 
        }
    ),
    
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            # Exclude latitude and longitude from being displayed to the client to make easier to read
            # These fields are still used internally for the geolocation map
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns if i not in ['location_lat', 'location_long']
        ],
        data=df.to_dict('records'),
        #Features for the interactive data table to make it user-friendly for the client
        row_selectable="single",       
        selected_rows=[0],            
        page_size=10,                  
        sort_action="native" 
    ),
    html.Br(),
    html.Hr(),
    #This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ]),
    
    # Add pink border on the bottom of the page
    html.Div(style={'borderTop': '10px solid #FFB6C1', 'margin': '20px 0'})
])

#############################################
# Interaction Between Components / Controller
#############################################

    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):     
    # If 'Reset' is selected, show all data
    if filter_type == 'reset':
        filtered_df = df  
    else:
        # Filter based on rescue type
        if filter_type == 'water':
            breeds = ['Labrador Retriever Mix', 'Chesa Bay Retr Mix', 'Newfoundland Mix']
            sex = 'Intact Female'
            min_age, max_age = 26, 156
        elif filter_type == 'mountain':
            breeds = ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog', 'Siberian Husky', 'Rottweiler']
            sex = 'Intact Male'
            min_age, max_age = 26, 156
        elif filter_type == 'disaster':
            breeds = ['Doberman Pinsch', 'Doberman Pinsch Mix', 'German Shepherd', 'Golden Retriever', 'Bloodhound', 'Rottweiler']
            sex = 'Intact Male'
            min_age, max_age = 20, 300
        
        # Filter the DataFrame based on the criteria
        filtered_df = df[ 
            (df['breed'].isin(breeds)) & 
            (df['sex_upon_outcome'] == sex) & 
            (df['age_upon_outcome_in_weeks'] >= min_age) & 
            (df['age_upon_outcome_in_weeks'] <= max_age)
        ]

    # Convert the filtered DataFrame to a list of dictionaries for DataTable
    return filtered_df.to_dict('records')  # Returning the filtered data

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('filter-type', 'value')]  # Add 'filter-type' as an input
)
def update_graphs(viewData, filter_type):  # Add filter_type as a parameter
    if viewData is None:
        return []
    
    # Convert viewData to DataFrame
    df = pd.DataFrame.from_dict(viewData)
    
    if filter_type == 'reset':
        # When reset is selected, show top 10 breeds in the pie chart
        breed_counts = df['breed'].value_counts()
        top_breeds = breed_counts.head(10)
        
        # If there are more than 10 breeds, group others as 'Other'
        if len(breed_counts) > 10:
            # Sum of all other breeds
            other_breeds = breed_counts.tail(len(breed_counts) - 10).sum()
            
            # Add 'Other' category
            top_breeds = pd.concat([top_breeds, pd.Series({'Other': other_breeds})])
        
        return [
            dcc.Graph(
                figure=px.pie(
                    names=top_breeds.index,
                    values=top_breeds.values,
                    title='Top 10 Preferred Animals (with Others)'
                )
            )
        ]
    
    # If filter is not reset, show the full dataset as a pie chart
    return [
        dcc.Graph(
            figure=px.pie(df, names='breed', title='Preferred Animals')
        )
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    # Added functionality to prevent the callback from throwing an error when the page first loads
    # If no columns are selected yet, set to empty list
    if selected_columns is None:
        selected_columns = []
        
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    # Check if no row is selected (index is None) and default to selecting the first row
    if index is None or len(index) == 0:  
        index = [0]  # Auto-select the first row if no row is selected
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Use column names instead of index numbers
            # This ensures the map continues to function even if column order changes
            dl.Marker(
                position=[dff.iloc[row]['location_lat'], dff.iloc[row]['location_long']],  
                children=[
                    # Tooltip shows the breed when hovering over the marker
                    dl.Tooltip(dff.iloc[row]['breed']),  
                    # Popup displays the animal's name when the marker is clicked
                    dl.Popup([
                        html.H1("Animal Name"),
                        html.P(dff.iloc[row]['name'])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Succesfully Connected
Dash app running on https://pupilpagoda-switchgossip-3000.codio.io/proxy/8050/
